# Pandas Hands-On Practice: E-Commerce Data Analysis

In this exercise, you will work with a **simulated e-commerce dataset** covering orders, customers, and products.

The data is intentionally messy — missing values, inconsistent types, duplicates — just like real-world data.

**What you will practice:**

| Task | Skills |
|------|--------|
| 1. Generate & Load Data | `pd.read_csv()`, `info()`, `describe()` |
| 2. Data Cleaning | `isna()`, `fillna()`, `dropna()`, `drop_duplicates()`, `astype()` |
| 3. Exploratory Data Analysis | `groupby()`, `value_counts()`, `sort_values()` |
| 4. Merging Tables | `pd.merge()` — inner, left, outer joins |
| 5. Pivot Tables | `pivot_table()`, `crosstab()` |
| 6. Simple Visualization | `df.plot()`, `matplotlib` basics |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Task 1: Generate the Simulated Data

We'll create **three CSV files** that mimic a real e-commerce database:
- `customers.csv` — customer profiles
- `products.csv` — product catalog
- `orders.csv` — transaction records (with missing values & duplicates)

In [ ]:
# ---- customers.csv ----
customers = pd.DataFrame({
    "customer_id": range(1001, 1011),
    "name": ["Alice", "Bob", "Charlie", "Diana", "Eve",
             "Frank", "Grace", "Henry", "Ivy", "Jack"],
    "city": ["New York", "Los Angeles", "Chicago", "New York", "San Francisco",
             "Chicago", "Los Angeles", "New York", "San Francisco", "Chicago"],
    "membership": ["Gold", "Silver", "Gold", "Bronze", "Gold",
                   "Silver", "Bronze", "Gold", "Silver", "Bronze"]
})
customers.to_csv("customers.csv", index=False)
print("customers.csv created")
customers

In [ ]:
# ---- products.csv ----
products = pd.DataFrame({
    "product_id": ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08"],
    "product_name": ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones",
                     "USB Cable", "Webcam", "Mousepad"],
    "category": ["Electronics", "Accessories", "Accessories", "Electronics",
                 "Electronics", "Accessories", "Electronics", "Accessories"],
    "unit_price": [999.99, 29.99, 79.99, 349.99, 149.99, 9.99, 69.99, 14.99]
})
products.to_csv("products.csv", index=False)
print("products.csv created")
products

In [ ]:
# ---- orders.csv (intentionally messy!) ----
n_orders = 200

order_ids = range(5001, 5001 + n_orders)
customer_ids = np.random.choice(range(1001, 1011), size=n_orders)
product_ids = np.random.choice(["P01","P02","P03","P04","P05","P06","P07","P08"], size=n_orders)
quantities = np.random.randint(1, 6, size=n_orders)

# Random dates in 2025
start = pd.Timestamp("2025-01-01")
dates = [start + pd.Timedelta(days=int(d)) for d in np.random.randint(0, 365, size=n_orders)]

orders = pd.DataFrame({
    "order_id": order_ids,
    "customer_id": customer_ids,
    "product_id": product_ids,
    "quantity": quantities,
    "order_date": dates
})

# --- Inject messiness ---

# 1) Missing values: ~8% of quantity, ~5% of customer_id, ~3% of product_id
qty_mask = np.random.random(n_orders) < 0.08
cid_mask = np.random.random(n_orders) < 0.05
pid_mask = np.random.random(n_orders) < 0.03
orders.loc[qty_mask, "quantity"] = np.nan
orders.loc[cid_mask, "customer_id"] = np.nan
orders.loc[pid_mask, "product_id"] = np.nan

# 2) Duplicate rows: copy 5 random rows
dup_rows = orders.sample(5, random_state=42)
orders = pd.concat([orders, dup_rows], ignore_index=True)

# 3) Shuffle so duplicates aren't at the bottom
orders = orders.sample(frac=1, random_state=7).reset_index(drop=True)

orders.to_csv("orders.csv", index=False)
print(f"orders.csv created: {len(orders)} rows (includes duplicates & missing values)")
orders.head(10)

---
## Task 2: Load & Inspect the Data

Now pretend you just received these files. Load them and get an overview.

**Your goals:**
- How many rows and columns in each table?
- What are the data types?
- How many missing values?

In [ ]:
# Load all three tables
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")
orders = pd.read_csv("orders.csv")

print(f"customers : {customers.shape}")
print(f"products  : {products.shape}")
print(f"orders    : {orders.shape}")

In [ ]:
# Inspect orders — the messy one
print("=== orders.info() ===")
orders.info()
print("\n=== orders.describe() ===")
orders.describe()

In [ ]:
# Count missing values per column
print("=== Missing values in orders ===")
print(orders.isna().sum())
print(f"\nTotal missing: {orders.isna().sum().sum()}")
print(f"Missing %: {(orders.isna().sum().sum() / orders.size * 100):.1f}%")

---
## Task 3: Data Cleaning

Real data is always messy. Let's clean it step by step:

1. **Remove duplicates**
2. **Handle missing values** — drop or fill depending on the column
3. **Fix data types** — `order_date` should be datetime, `customer_id` should be int

### 3.1 Remove Duplicates

In [ ]:
# Check for duplicates
n_dup = orders.duplicated().sum()
print(f"Duplicate rows found: {n_dup}")

# Show the duplicates
print("\nDuplicate rows:")
orders[orders.duplicated(keep=False)].sort_values("order_id")

In [ ]:
# Remove duplicates — keep the first occurrence
orders_clean = orders.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Before: {len(orders)} rows → After: {len(orders_clean)} rows")

### 3.2 Handle Missing Values

In [ ]:
# Check what's missing
print("Missing values after dedup:")
print(orders_clean.isna().sum())

In [ ]:
# Strategy:
# - customer_id missing → we can't identify the buyer → DROP these rows
# - product_id missing  → we can't identify the item  → DROP these rows
# - quantity missing     → fill with the median (reasonable assumption for small gaps)

# Step 1: Drop rows where customer_id or product_id is missing
before = len(orders_clean)
orders_clean = orders_clean.dropna(subset=["customer_id", "product_id"])
print(f"Dropped {before - len(orders_clean)} rows with missing customer_id or product_id")

# Step 2: Fill missing quantity with median
median_qty = orders_clean["quantity"].median()
print(f"Median quantity: {median_qty}")
orders_clean["quantity"] = orders_clean["quantity"].fillna(median_qty)

# Verify: no more missing values
print(f"\nRemaining missing values: {orders_clean.isna().sum().sum()}")

### 3.3 Fix Data Types

In [ ]:
# Check current types
print("Before:")
print(orders_clean.dtypes)

# Fix: customer_id float → int, quantity float → int, order_date str → datetime
orders_clean["customer_id"] = orders_clean["customer_id"].astype(int)
orders_clean["quantity"] = orders_clean["quantity"].astype(int)
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])

print("\nAfter:")
print(orders_clean.dtypes)

# Reset index for a clean table
orders_clean = orders_clean.reset_index(drop=True)
print(f"\nClean dataset: {orders_clean.shape}")
orders_clean.head()

---
## Task 4: Merge Tables

Now let's combine the three tables to get a **complete picture** of each order.

We need:
- Order details (from `orders_clean`)
- Customer name & city (from `customers`)
- Product name, category & price (from `products`)

This is exactly what `pd.merge()` does — like SQL JOIN.

In [ ]:
# Step 1: Merge orders with customers on customer_id
df = pd.merge(orders_clean, customers, on="customer_id", how="left")
print(f"After merging customers: {df.shape}")
df.head()

In [ ]:
# Step 2: Merge with products on product_id
df = pd.merge(df, products, on="product_id", how="left")
print(f"After merging products: {df.shape}")
df.head()

In [ ]:
# Step 3: Calculate total_amount = quantity × unit_price
df["total_amount"] = df["quantity"] * df["unit_price"]

# Reorder columns for readability
df = df[["order_id", "order_date", "name", "city", "membership",
         "product_name", "category", "quantity", "unit_price", "total_amount"]]

print(f"Final merged table: {df.shape}")
df.head(10)

---
## Task 5: Exploratory Data Analysis (EDA)

Now we have a clean, merged dataset. Let's answer some business questions:

1. What is the **total revenue**?
2. Which **products** sell the most?
3. Which **cities** generate the most revenue?
4. Do **Gold members** spend more than others?
5. What are the **monthly trends**?

### 5.1 Overall Summary

In [ ]:
print(f"Total orders  : {len(df)}")
print(f"Total revenue : ${df['total_amount'].sum():,.2f}")
print(f"Avg order size: ${df['total_amount'].mean():,.2f}")
print(f"Date range    : {df['order_date'].min().date()} → {df['order_date'].max().date()}")
print(f"\nUnique customers : {df['name'].nunique()}")
print(f"Unique products  : {df['product_name'].nunique()}")

### 5.2 Top Products by Revenue

In [ ]:
# Group by product, sum revenue, sort descending
product_revenue = (df.groupby("product_name")["total_amount"]
                     .sum()
                     .sort_values(ascending=False))

print("=== Revenue by Product ===")
print(product_revenue.apply(lambda x: f"${x:,.2f}"))

### 5.3 Revenue by City

In [ ]:
# Group by city: total revenue and order count
city_stats = (df.groupby("city")
                .agg(total_revenue=("total_amount", "sum"),
                     order_count=("order_id", "count"),
                     avg_order=("total_amount", "mean"))
                .sort_values("total_revenue", ascending=False))

city_stats["total_revenue"] = city_stats["total_revenue"].map("${:,.2f}".format)
city_stats["avg_order"] = city_stats["avg_order"].map("${:,.2f}".format)
print("=== Revenue by City ===")
city_stats

### 5.4 Spending by Membership Level

In [ ]:
# Do Gold members spend more per order?
membership_stats = (df.groupby("membership")
                      .agg(total_revenue=("total_amount", "sum"),
                           order_count=("order_id", "count"),
                           avg_order=("total_amount", "mean"))
                      .sort_values("total_revenue", ascending=False))

print("=== Spending by Membership ===")
membership_stats

### 5.5 Monthly Revenue Trend

In [ ]:
# Extract month from order_date
df["month"] = df["order_date"].dt.to_period("M")

monthly = (df.groupby("month")
             .agg(revenue=("total_amount", "sum"),
                  orders=("order_id", "count"))
             .reset_index())

# Convert Period to string for display
monthly["month"] = monthly["month"].astype(str)
print("=== Monthly Revenue ===")
monthly

---
## Task 6: Pivot Tables

Pivot tables reshape your data for cross-tabular analysis — like Excel pivot tables.

**Question:** What is the total revenue for each product category in each city?

In [ ]:
# Pivot table: rows = city, columns = category, values = total revenue
pivot = pd.pivot_table(
    df,
    values="total_amount",     # what to aggregate
    index="city",              # row labels
    columns="category",        # column labels
    aggfunc="sum",             # aggregation function
    margins=True,              # add row/column totals
    margins_name="Total"       # label for the totals
)

print("=== Revenue: City × Category ===")
pivot.applymap(lambda x: f"${x:,.0f}")

In [ ]:
# Pivot table: average quantity per membership level × product
pivot2 = pd.pivot_table(
    df,
    values="quantity",
    index="membership",
    columns="product_name",
    aggfunc="mean"
).round(1)

print("=== Avg Quantity: Membership × Product ===")
pivot2

In [ ]:
# crosstab: count orders per city × membership
ct = pd.crosstab(df["city"], df["membership"], margins=True)
print("=== Order Count: City × Membership ===")
ct

---
## Task 7: Simple Visualization

Pandas has built-in plotting (powered by `matplotlib`). Let's make a few quick charts.

> We will cover visualization in detail in a later lecture. For now, just see how easy it is!

In [ ]:
# Bar chart: Revenue by product
product_rev = df.groupby("product_name")["total_amount"].sum().sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
product_rev.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Total Revenue ($)")
ax.set_title("Revenue by Product")
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart: Order share by city
city_orders = df["city"].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
city_orders.plot.pie(ax=ax, autopct="%1.1f%%", startangle=90)
ax.set_ylabel("")  # remove default ylabel
ax.set_title("Order Share by City")
plt.show()

In [ ]:
# Line chart: Monthly revenue trend
monthly_rev = df.groupby(df["order_date"].dt.to_period("M"))["total_amount"].sum()
monthly_rev.index = monthly_rev.index.astype(str)  # for clean x-axis labels

fig, ax = plt.subplots(figsize=(10, 4))
monthly_rev.plot(ax=ax, marker="o", color="coral", linewidth=2)
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($)")
ax.set_title("Monthly Revenue Trend (2025)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [1]:
# Stacked bar: Revenue by city and category
pivot_plot = pd.pivot_table(df, values="total_amount", index="city", columns="category", aggfunc="sum")

fig, ax = plt.subplots(figsize=(8, 5))
pivot_plot.plot.bar(ax=ax, stacked=True, colormap="Set2")
ax.set_ylabel("Revenue ($)")
ax.set_title("Revenue by City and Category")
plt.xticks(rotation=0)
plt.legend(title="Category")
plt.tight_layout()
plt.show()

NameError: name 'pd' is not defined

---
## Summary

In this exercise, you practiced the **complete data analysis workflow**:

```
Raw CSV files
    │
    ├── 1. Load & Inspect    →  read_csv(), info(), describe()
    ├── 2. Clean             →  drop_duplicates(), dropna(), fillna(), astype()
    ├── 3. Merge             →  pd.merge() — combine related tables
    ├── 4. Explore (EDA)     →  groupby(), value_counts(), sort_values()
    ├── 5. Pivot             →  pivot_table(), crosstab()
    └── 6. Visualize         →  df.plot(), matplotlib
```

**Key takeaways:**
- Always inspect your data first — `info()` and `isna().sum()` are your best friends
- Decide a cleaning strategy per column: drop vs. fill, and document why
- `pd.merge()` is the Pandas equivalent of SQL JOIN — learn the `how` parameter
- `pivot_table()` is incredibly powerful for cross-tabular summaries
- A simple chart is worth a thousand `print()` statements